# 05 学生向け研究実験テンプレート

## 1. 研究目的

チームで研究テーマを選び、**自分たちで実験を追加する**ためのテンプレートです。**完成した研究結果は入っていません。**

| Research | テーマ | 区分 | 関連 Notebook |
| --- | --- | --- | --- |
| A | Python と NumPy | Core | 01 |
| B | CPU と GPU | Core（GPU がなければ CPU 部分のみ） | 02 |
| C | CPU・GPU・TPU（同じ JAX 関数で比較） | Advanced | 03 |
| D | アルゴリズムと計算量 | Core | 04 |
| E | メモリと計算性能 | Advanced | 01・03 |

## 2. 理論的背景（研究の進め方）

- **仮説を先に書く**。「なぜそうなると思うか」も書く。
- **一度に 1 つの条件だけを変える**（他は固定する）。
- **記録する**：問題サイズ、dtype、乱数シード、反復回数とウォームアップ、統計量（中央値など）、実行環境（ランタイム、CPU・GPU・TPU の名前）。
- **正しさを先に確認する**（`np.allclose` など）。速くても結果が違えば意味がない。
- **測っていないものを書かない**。使えなかったデバイスの値を推測で埋めない。
- **理論計算量**（オーダー）と**実測時間**（定数倍・ハードウェア・実装）を区別する。

## 3. 実験環境（最初に必ず実行）

このセルは、E2 リポジトリの `.py`（`cpu_basics.py`、`numpy_benchmark.py` など）を import できるように、
**作業ディレクトリ（E2 のルート）を自動で探して設定**します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのチームのリポジトリの URL** を入れる（`REPO_SUBDIR` に E2 のフォルダ名） |
| Colab（方法B：ZIP をアップロード） | E2 フォルダを ZIP にして `/content` にアップロードしておく（またはこのセルの実行中にアップロード） |

> ⚠️ `textbook/`（教科書のコード）は**公開リポジトリには含めない**約束です。
> `textbook/` がないと `algorithm_benchmark.py` の import が失敗します（Notebook 04・05 の一部）。
> その場合は方法B（ZIP）を使うか、授業で配布された `textbook/` を同じ場所に置いてください。

In [ ]:
import os
import sys
import glob
import zipfile
import subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                              # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "e2-computing-performance"   # リポジトリの中の E2 フォルダ（ルートが E2 なら "" にする）

IN_COLAB = "google.colab" in sys.modules


def is_e2_root(p):
    """cpu_basics.py と benchmark_utils.py があるフォルダを E2 のルートとみなす"""
    return (Path(p) / "cpu_basics.py").exists() and (Path(p) / "benchmark_utils.py").exists()


def find_e2_root():
    # 1) ローカル：notebooks/ の中、または E2 のルートで開いた場合
    for p in [Path.cwd(), Path.cwd().parent]:
        if is_e2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    # 2) Colab 方法A：clone
    if REPO_URL:
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_e2_root(cand):
            return cand
    # 3) Colab 方法B：/content にある ZIP を展開（なければアップロードを求める）
    zips = glob.glob("/content/*.zip")
    if not zips:
        from google.colab import files
        print("E2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/e2_upload")
    for hit in glob.glob("/content/e2_upload/**/cpu_basics.py", recursive=True):
        if is_e2_root(Path(hit).parent):
            return Path(hit).parent
    return None


E2_ROOT = find_e2_root()
if E2_ROOT is None:
    raise RuntimeError("E2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(E2_ROOT)
if str(E2_ROOT) not in sys.path:
    sys.path.insert(0, str(E2_ROOT))
print("E2_ROOT:", E2_ROOT)
if not (E2_ROOT / "textbook" / "bsearch.py").exists():
    print("⚠️ textbook/ がありません。algorithm_benchmark.py の import が失敗します（上の注意を参照）。")

In [ ]:
import matplotlib.pyplot as plt

# グラフの共通スタイル（Colab には日本語フォントがないことが多いので、図の文字は英語にしている）
plt.rcParams.update({
    "figure.figsize": (7, 3.6), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
})
# 色は「実装（ライブラリ）」ごとに固定する。デバイス（CPU / GPU / TPU）は線の種類と記号で区別する
COLORS = {"Python": "#eb6834", "NumPy": "#2a78d6", "PyTorch": "#1baf7a", "JAX": "#6250d6",
          "TensorFlow": "#eda100", "neutral": "#8a8985"}
DEVICE_STYLE = {"CPU": ("-", "o"), "GPU": ("--", "s"), "TPU": ("-.", "^")}

In [ ]:
import platform

def show_environment():
    """Python・ライブラリのバージョン、CPU、GPU・TPU の有無を表示する（測定はしない）"""
    info = {"Python": platform.python_version(), "Platform": platform.platform()}
    cpu = platform.processor() or platform.machine()
    try:
        with open("/proc/cpuinfo") as f:
            cpu = next(line.split(":", 1)[1].strip() for line in f if line.startswith("model name"))
    except (OSError, StopIteration):
        pass
    info["CPU"] = f"{cpu} ({os.cpu_count()} logical cores)"
    try:
        import numpy
        info["NumPy"] = numpy.__version__
    except ImportError:
        info["NumPy"] = "not installed"
    try:
        import torch
        info["PyTorch"] = torch.__version__
        info["CUDA GPU"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available"
    except ImportError:
        info["PyTorch"] = "not installed"
        info["CUDA GPU"] = "unknown (PyTorch not installed)"
    try:
        import jax
        info["JAX"] = jax.__version__
        info["JAX devices"] = ", ".join(f"{d.platform}:{d.device_kind}" for d in jax.devices())
        info["TPU"] = "available" if any(d.platform == "tpu" for d in jax.devices()) else "not available"
    except ImportError:
        info["JAX"] = "not installed"
        info["TPU"] = "unknown (JAX not installed)"
    except Exception as e:   # TPU の初期化に失敗した場合など
        info["JAX"] = info.get("JAX", "installed") + f" (device error: {e})"
        info["TPU"] = "not available"
    for k, v in info.items():
        print(f"{k:12}: {v}")
    return info

ENV = show_environment()

## 4. 使用するコード

既存の `.py` を import します。新しく書いた関数のうち再利用できるものは `.py` に移し、`tests/` にテストを追加してください。

In [ ]:
import sys
import time
import numpy as np

import cpu_basics as cb
import numpy_benchmark as nb
from benchmark_utils import measure, format_table, save_csv

try:
    import algorithm_benchmark as ab        # textbook/ が必要
except ImportError as e:
    ab = None
    print("algorithm_benchmark を import できません（textbook/ を確認）:", e)

SEED = 0          # TODO：使った乱数シードを報告に書く
REPEATS = 5
os.makedirs("notebooks/results", exist_ok=True)

## 5〜7. 実験・可視化・解釈 ―― 各 Research で自分たちで行う

各 Research には、**最低限の材料（変数・関数の呼び出し例）と TODO** だけを置いています。セルはそのまま実行してもエラーになりませんが、研究の結果は出しません。

---
## Research A：Python と NumPy（Core）

**目的**：データサイズ・dtype・配列の形状によって、NumPy の高速化率とメモリ使用量がどう変わるかを調べる。

**調べること**
- データサイズと高速化率
- dtype（float64・float32・int64）の影響
- 配列の形状（1 次元・2 次元、行方向・列方向の和）
- メモリ使用量（list と配列）

| 項目 | 記入欄 |
| --- | --- |
| 仮説（実験の前に書く） | TODO |
| 操作変数 | TODO |
| 評価指標 | TODO |
| 固定する条件（dtype・乱数シード・反復回数・ランタイム） | TODO |
| 担当者 | TODO |

In [ ]:
sizes_A = [100, 1_000, 10_000]          # TODO：範囲を決める
dtypes_A = [np.float64, np.float32]     # TODO：int64 なども
print(nb.compare("elementwise", sizes_A[0], repeats=REPEATS, dtype=dtypes_A[0]))   # 例：1 条件だけ

# TODO A-1：sizes_A × dtypes_A × task で speedup の表を作る
# TODO A-2：2 次元配列 M（例 1000×1000）で M.sum(axis=0) と M.sum(axis=1) の時間を比べる（メモリの並び順）
# TODO A-3：list と配列のメモリ使用量（sys.getsizeof、arr.nbytes）をサイズごとに表にする
results_A = []

**結果**（表・グラフ）：TODO

**考察**（仮説は正しかったか、なぜそうなったか）：TODO

**限界**（この実験から言えないこと）：TODO

---
## Research B：CPU と GPU（Core。GPU がない場合は CPU の NumPy と PyTorch の比較を行う）

**目的**：行列サイズ・バッチサイズ・データ転送・ウォームアップが、CPU と GPU の比較にどう影響するかを調べ、GPU が有利になる条件を求める。

**調べること**
- 行列サイズ
- バッチサイズ
- データ転送（転送込み／計算本体のみ）
- ウォームアップの有無
- GPU が有利になる条件

| 項目 | 記入欄 |
| --- | --- |
| 仮説（実験の前に書く） | TODO |
| 操作変数 | TODO |
| 評価指標 | TODO |
| 固定する条件（dtype・乱数シード・反復回数・ランタイム） | TODO |
| 担当者 | TODO |

In [ ]:
try:
    import torch
    devices_B = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
except ImportError:
    torch, devices_B = None, []
print("PyTorch devices:", devices_B or "PyTorch なし")

matrix_sizes_B = [64, 256, 1024]   # TODO
# TODO B-1：notebooks/02 の bench_matmul を参考に、サイズごとに CPU と GPU の compute_s を測る
# TODO B-2：to_device_s + compute_s + to_host_s（転送込み）でも比べる
# TODO B-3：ウォームアップなし（warmup=0）で 1 回目だけを測った場合と比べる
# TODO B-4：GPU が CPU より速くなる行列サイズの境目を求める（GPU がない場合はその旨を記録）
results_B = []

**結果**（表・グラフ）：TODO

**考察**（仮説は正しかったか、なぜそうなったか）：TODO

**限界**（この実験から言えないこと）：TODO

---
## Research C：CPU・GPU・TPU（Advanced）

**目的**：同じ JAX 関数を異なるデバイスで実行し、初回コンパイル・JIT の効果・問題サイズ・デバイスの違いを分析する。

**調べること**
- 同じ JAX 関数での比較
- 初回コンパイル時間
- JIT の効果（jit あり／なし）
- 問題サイズ
- 実行デバイスの違い

| 項目 | 記入欄 |
| --- | --- |
| 仮説（実験の前に書く） | TODO |
| 操作変数 | TODO |
| 評価指標 | TODO |
| 固定する条件（dtype・乱数シード・反復回数・ランタイム） | TODO |
| 担当者 | TODO |

In [ ]:
try:
    import jax
    import jax.numpy as jnp
    print("JAX devices:", jax.devices())
except ImportError:
    jax = None
    print("JAX なし（Colab では標準で入っています）")

# TODO C-1：notebooks/03 を CPU・GPU・TPU の各ランタイムで実行し、保存した CSV を読み込む
csv_files_C = sorted(glob.glob("notebooks/results/03_cpu_gpu_tpu_*.csv"))
print("CSV:", csv_files_C or "まだありません（notebooks/03 を実行して保存してください）")
# TODO C-2：同じ op・n で、デバイスごとの compute_s を比べる
# TODO C-3：first_call_s / compute_s を計算し、「コンパイルの元が取れる回数」を求める
# TODO C-4：jit なし・jit ありで、演算をいくつも組み合わせた関数の時間を比べる
# TODO C-5：n を変えたとき、デバイスの順位が入れ替わるか調べる
results_C = []

**結果**（表・グラフ）：TODO

**考察**（仮説は正しかったか、なぜそうなったか）：TODO

**限界**（この実験から言えないこと）：TODO

---
## Research D：アルゴリズムと計算量（Core）

**目的**：教科書の探索・ソートについて、入力データの性質と問題サイズを変え、理論計算量と実測時間を比べる。

**調べること**
- 線形探索と二分探索（ソートの時間を含む／含まない）
- ソートアルゴリズム
- 入力データの性質（ソート済み・逆順・重複）
- 理論計算量と実測時間

| 項目 | 記入欄 |
| --- | --- |
| 仮説（実験の前に書く） | TODO |
| 操作変数 | TODO |
| 評価指標 | TODO |
| 固定する条件（dtype・乱数シード・反復回数・ランタイム） | TODO |
| 担当者 | TODO |

In [ ]:
if ab is not None:
    print(ab.sort_experiment(200, "random", names=["insertion_sort", "sorted"], repeats=2, seed=SEED))  # 例

# TODO D-1：n を変えて search_experiment を実行し、線形探索と二分探索を表・グラフにする
# TODO D-2：make_data の order を変えて、各ソートの最良・最悪の場合を調べる
# TODO D-3：complexity_table で時間 / f(n) を計算し、理論どおりか確かめる
# TODO D-4：教科書の他のソート（selection_sort、quick_sort など）を追加する場合は、
#           原本を textbook/ にコピーして SHA-256 で一致を確認する
results_D = []

**結果**（表・グラフ）：TODO

**考察**（仮説は正しかったか、なぜそうなったか）：TODO

**限界**（この実験から言えないこと）：TODO

---
## Research E：メモリと計算性能（Advanced）

**目的**：配列サイズ・dtype・メモリアクセスのパターンが、メモリ使用量と計算時間にどう影響するかを調べる。

**調べること**
- 配列サイズ（CPU のキャッシュに入る大きさ／入らない大きさ）
- dtype とメモリ使用量
- メモリアクセス（連続・飛び飛び）
- 計算時間（GB/s に換算）

| 項目 | 記入欄 |
| --- | --- |
| 仮説（実験の前に書く） | TODO |
| 操作変数 | TODO |
| 評価指標 | TODO |
| 固定する条件（dtype・乱数シード・反復回数・ランタイム） | TODO |
| 担当者 | TODO |

In [ ]:
def bandwidth_gbs(n_bytes, seconds):
    """読み書きしたバイト数と時間から、見かけのメモリ帯域（GB/s）を計算する"""
    return n_bytes / seconds / 1e9

x = np.ones(1_000_000, dtype=np.float32)
t = measure(lambda: x * 2.0, repeats=REPEATS, warmup=1)["median"]
print(f"x * 2.0 (n = {x.size:,}, {x.dtype}): {t * 1e3:.3f} ms, "
      f"~{bandwidth_gbs(2 * x.nbytes, t):.1f} GB/s（読み + 書き）")   # 例：1 条件だけ

# TODO E-1：n を 1e3〜1e8 程度まで変えて GB/s を測り、キャッシュの大きさで変化するか調べる（メモリ上限に注意）
# TODO E-2：dtype（float64・float32・float16）で時間と GB/s を比べる
# TODO E-3：x[::k]（飛び飛びのアクセス）で、k を変えたときの 1 要素あたりの時間を比べる
# TODO E-4：（GPU）転送のバイト数と時間から、CPU→GPU の転送帯域を求める
results_E = []

**結果**（表・グラフ）：TODO

**考察**（仮説は正しかったか、なぜそうなったか）：TODO

**限界**（この実験から言えないこと）：TODO

## 8. 学生向け TODO（まとめ）

- [ ] 取り組む Research を選び（Core から 1 つ以上、Advanced は任意）、仮説・操作変数・評価指標を記入する
- [ ] 各 Research の TODO を実装する
- [ ] 結果を `save_csv(rows, "notebooks/results/xxx.csv")` で保存し、表・グラフにまとめる
- [ ] 結果の解釈と限界を書く
- [ ] 再利用できる関数は `.py` に移し、`tests/` にテストを追加する（`python3 -m unittest discover -s tests -v`）

## 9. 考察課題

1. RQ1〜RQ4（README 1 章）に、自分たちの実験からどこまで答えられましたか。
2. 「理論計算量が同じでも実行時間が違う」例と、「実装を速くしてもオーダーは変わらない」例を、自分たちの結果から 1 つずつ挙げてください。
3. 自分たちの比較は「デバイスの差」「ライブラリの差」「アルゴリズムの差」のどれを見ていますか。混ざっている場合、どう切り分けられますか。

## 10. 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 探索、第6章 ソート）
- 参考 Notebook：`ai-programming-2026-team-project-1-starter/A_cpu_gpu_tpu/A_cpu_gpu_tpu_benchmark/starter.ipynb`（MIT License, Copyright (c) 2026 Daigo Takeuchi）
- NumPy ドキュメント <https://numpy.org/doc/>
- PyTorch ドキュメント：CUDA semantics（非同期実行と同期）<https://pytorch.org/docs/stable/notes/cuda.html>
- JAX ドキュメント：Asynchronous dispatch、Just-in-time compilation <https://docs.jax.dev/>

- E2 リポジトリの `README.md`